In [1]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Importamos los Datos

In [2]:
# Abre el archivo en modo lectura binaria ('rb')
with open('../data/bank_26.pkl', 'rb') as archivo:
    df = pickle.load(archivo)

C:\Users\aleja\AppData\Local\Temp\ipykernel_2252\3242033964.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df = pickle.load(archivo)


# Preprocesado

#### Pdays
Para empezar con el preprocesado, primero vamos a hacer un poco de feature engineering con la variable pdays

In [ ]:
# Pdays preprocessing

# Función para el preprocesado de los pdays
def process_pdays(X):
    X_out = X.copy()
    X_out['contacto_previo'] = np.where(X_out['pdays'] == -1, 0, 1)
    X_out['pdays'] = X_out['pdays'].replace(-1, 0)
    return X_out

# Lo convertimos en un transformer para añadirlo al pipeline más adelante
pdays_transformer = FunctionTransformer(process_pdays)

#### Eliminación de variables
Una parte del preprocesado y feature engineering es la eliminación de variables que no ayudan al modelo o que pueden producir data leakage. En este caso la variable 'duration', que representa la duración de la llamada con el cliente, no tiene mucho sentido tenerla ya que el modelo trata de predecir clientes a los que realizar una llamada, por ende la llamada no se ha realizado cuando hagamos la predicción y no se podrá tener en cuenta.

In [ ]:
def drop_columns(X):
    return X.drop(['duration'], axis=1, errors='ignore') 

drop_transformer = FunctionTransformer(drop_columns)

Una vez realizado el preprocesado y el feature engineering, procedemos al encoding y escalado final. Dividimos el set entre columnas numéricas y categóricas. A las numéricas le aplicamos un escalado estándar, y a las categóricas aplicamos un simple imputer (para eliminar valores nulos en caso de que los siga habiendo), y un OneHotEncoder para transformar en variables binarias las categóricas.

In [ ]:
# Final encoder

num_cols = ['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous', 'contacto_previo']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Transformer para variables categóricas (Imputación + Encoding)
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='if_binary')) 
])

# Unimos ambos transformers en un ColumnTransformer
final_encoder = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

Generamos el Pipeline de preprocesado (que luego será incluido en el pipeline final, con el modelo incluido). Hemos diseñado esta arquitectura con tantos Pipelines ya que es la forma más modular para trabajar. Si quisiésemos añadir una función de preprocesado nueva, tan solo habría que escribir la función y meterla en el Pipeline, sin estar preocupados de si hay data leakage, de en que parte del código se realiza...

In [ ]:
pre_processing = Pipeline(steps=[
    ("pdays", pdays_transformer),      # Arreglamos los -1 y creamos la variable binaria
    ("drop", drop_transformer),        # Dropeamos columnas problemáticas (duration)
    ("encoder", final_encoder)         # Imputamos nulos, codificamos texto y escalamos números
])

Separamos el set de datos entre datos de train y datos de test

In [ ]:
from sklearn.model_selection import train_test_split

# Separamos X e y
X = df.drop('deposit', axis=1)
y = df['deposit'].map({'yes': 1, 'no': 0})

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=100499978, stratify=y)